# Hair Extraction & Visualization
Extract and display hair region from face parsing segmentation.
Uses locally downloaded face parsing model.

## 1. Setup & Dependencies

In [1]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
from PIL import Image
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
)

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

c:\Users\PiyushChunara\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.0
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Using device: cpu


## 2. Load Locally Downloaded Face Parsing Model

In [4]:
# Path to locally downloaded model
model_path = "../face-parsing"  # Change to your model folder path if different

if os.path.exists(model_path):
    print(f"Loading model from: {model_path}")
    processor = SegformerImageProcessor.from_pretrained(model_path)
    model = SegformerForSemanticSegmentation.from_pretrained(model_path)
    print("✓ Face Parsing Model Loaded Successfully")
else:
    print(f"Model path not found: {model_path}")
    print("Please update model_path to your downloaded face_parsing folder")

model.to(device)
model.eval()
print("Model moved to device and set to evaluation mode")

Loading model from: ../face-parsing


ImportError: 
SegformerForSemanticSegmentation requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


## 3. Load Image

In [ ]:
# Path to input image
image_path = "side.png"  # Replace with your image path

if os.path.exists(image_path):
    image = Image.open(image_path).convert("RGB")
    img_rgb = np.array(image)
    print(f"✓ Image loaded: {image.size}")
    
    # Display original image
    plt.figure(figsize=(8, 8))
    plt.imshow(img_rgb)
    plt.title("Original Image", fontsize=14, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(f"Image not found at: {image_path}")
    print("Please provide a valid image path")

## 4. Run Face Segmentation

In [ ]:
# Run segmentation
inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

# Upsample to original image size
upsampled_logits = torch.nn.functional.interpolate(
    logits,
    size=image.size[::-1],
    mode="bilinear",
    align_corners=False,
)

labels = upsampled_logits.argmax(dim=1)[0]
labels = labels.cpu().numpy()

print(f"✓ Segmentation complete")
print(f"Unique labels detected: {np.unique(labels)}")

## 5. Define Facial Part Labels

In [ ]:
LABEL_MAP = {
    0: "background",
    1: "skin",
    2: "nose",
    3: "eye_g",
    4: "left_eye",
    5: "right_eye",
    6: "left_eyebrow",
    7: "right_eyebrow",
    8: "left_ear",
    9: "right_ear",
    10: "mouth",
    11: "upper_lip",
    12: "lower_lip",
    13: "hair",
    14: "hat",
    15: "earring",
    16: "necklace",
    17: "neck",
    18: "cloth",
}

print(f"Total facial regions: {len(LABEL_MAP)}")

## 6. Extract Hair Region

In [ ]:
# Hair label ID is 13
HAIR_LABEL = 13

# Create hair mask
hair_mask = (labels == HAIR_LABEL)

# Statistics
hair_pixel_count = np.sum(hair_mask)
total_pixels = hair_mask.size
hair_coverage = (hair_pixel_count / total_pixels * 100)

print(f"Hair Extraction Results:")
print(f"  Hair pixels: {hair_pixel_count}")
print(f"  Total pixels: {total_pixels}")
print(f"  Hair coverage: {hair_coverage:.2f}%")

## 7. Visualize Hair Mask

In [ ]:
# Display hair mask
plt.figure(figsize=(10, 8))
plt.imshow(hair_mask, cmap="gray")
plt.colorbar(label="Hair Region", shrink=0.8)
plt.title(f"Hair Segmentation Mask ({hair_coverage:.2f}% coverage)", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Hair Extraction - Method 1: White Background

In [ ]:
# Extract hair with white background
hair_white_bg = np.ones_like(img_rgb) * 255
hair_white_bg[hair_mask] = img_rgb[hair_mask]

# Display
plt.figure(figsize=(10, 8))
plt.imshow(hair_white_bg)
plt.title("Hair Extraction - White Background", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

## 9. Hair Extraction - Method 2: Transparent Background

In [ ]:
# Extract hair with transparent background (RGBA)
hair_rgba = np.zeros((img_rgb.shape[0], img_rgb.shape[1], 4), dtype=np.uint8)
hair_rgba[hair_mask, :3] = img_rgb[hair_mask]  # RGB channels
hair_rgba[hair_mask, 3] = 255  # Alpha channel (opaque where hair)

# Display (with checkerboard pattern to show transparency)
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(hair_rgba[:, :, :3])  # Show RGB only
ax.set_title("Hair Extraction - Transparent Background (RGB shown)", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

print("Note: Alpha channel (transparency) is present in the RGBA image")

## 10. Hair Extraction - Method 3: Cropped to Bounding Box

In [ ]:
# Find bounding box
ys, xs = np.where(hair_mask)

if len(xs) > 0:
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    
    print(f"Hair bounding box: ({x1}, {y1}) to ({x2}, {y2})")
    print(f"Bounding box size: {x2-x1} x {y2-y1} pixels")
    
    # Crop image and mask
    hair_cropped = img_rgb[y1:y2+1, x1:x2+1].copy()
    hair_mask_cropped = hair_mask[y1:y2+1, x1:x2+1]
    
    # Add white background
    hair_cropped_white_bg = np.ones_like(hair_cropped) * 255
    hair_cropped_white_bg[hair_mask_cropped] = hair_cropped[hair_mask_cropped]
    
    # Display
    plt.figure(figsize=(10, 8))
    plt.imshow(hair_cropped_white_bg)
    plt.title(f"Hair Extraction - Cropped (Size: {hair_cropped.shape[1]}x{hair_cropped.shape[0]})", 
             fontsize=14, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No hair detected in image")

## 11. Comparison: All Extraction Methods

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Original image
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title("Original Image", fontsize=13, fontweight="bold")
axes[0, 0].axis("off")

# Hair mask
axes[0, 1].imshow(hair_mask, cmap="gray")
axes[0, 1].set_title("Hair Mask (Binary)", fontsize=13, fontweight="bold")
axes[0, 1].axis("off")

# Hair with white background
axes[1, 0].imshow(hair_white_bg)
axes[1, 0].set_title("Hair - White Background (Full Size)", fontsize=13, fontweight="bold")
axes[1, 0].axis("off")

# Hair cropped
if len(xs) > 0:
    axes[1, 1].imshow(hair_cropped_white_bg)
    axes[1, 1].set_title("Hair - Cropped (White Background)", fontsize=13, fontweight="bold")
else:
    axes[1, 1].text(0.5, 0.5, "No hair detected", ha="center", va="center", fontsize=12)
    axes[1, 1].set_title("Hair - Cropped", fontsize=13, fontweight="bold")

axes[1, 1].axis("off")

plt.suptitle(f"Hair Extraction Comparison (Coverage: {hair_coverage:.2f}%)", 
             fontsize=15, fontweight="bold", y=0.995)
plt.tight_layout()
plt.show()

## 12. Hair Statistics & Analysis

In [ ]:
print("="*70)
print("HAIR EXTRACTION STATISTICS")
print("="*70)
print(f"\nImage Dimensions:")
print(f"  Width:  {img_rgb.shape[1]} pixels")
print(f"  Height: {img_rgb.shape[0]} pixels")
print(f"  Total pixels: {total_pixels:,}")

print(f"\nHair Region:")
print(f"  Hair pixels: {hair_pixel_count:,}")
print(f"  Coverage: {hair_coverage:.2f}%")

if len(xs) > 0:
    print(f"\nBounding Box:")
    print(f"  Top-Left: ({x1}, {y1})")
    print(f"  Bottom-Right: ({x2}, {y2})")
    print(f"  Width: {x2-x1} pixels")
    print(f"  Height: {y2-y1} pixels")
    print(f"  Area: {(x2-x1)*(y2-y1):,} pixels")

print("\nExtraction Methods Available:")
print("  1. White Background (Full Size)")
print("  2. Transparent Background (RGBA)")
print("  3. Cropped to Bounding Box")
print("\n" + "="*70)

## 13. Visualize Hair Overlaid on Original

In [ ]:
# Create overlay showing detected hair region
overlay = img_rgb.copy().astype(float)

# Highlight hair region in red
overlay[hair_mask, 0] = 255  # Red channel
overlay[hair_mask, 1] = overlay[hair_mask, 1] * 0.5  # Reduce green
overlay[hair_mask, 2] = overlay[hair_mask, 2] * 0.5  # Reduce blue

overlay = overlay.astype(np.uint8)

# Blend original and overlay
blended = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(img_rgb)
axes[0].set_title("Original Image", fontsize=13, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(blended)
axes[1].set_title("Hair Region Highlighted (Red Overlay)", fontsize=13, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 14. All Facial Parts Comparison

In [ ]:
# Show segmentation map with all labels
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Full segmentation map
im1 = axes[0].imshow(labels, cmap="nipy_spectral")
axes[0].set_title("Face Parsing Segmentation Map (All Parts)", fontsize=13, fontweight="bold")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], label="Label ID")

# Show individual parts detected
detected_parts = []
for label_id in np.unique(labels):
    part_name = LABEL_MAP.get(label_id, f"Unknown {label_id}")
    pixel_count = np.sum(labels == label_id)
    percentage = (pixel_count / total_pixels * 100)
    detected_parts.append((label_id, part_name, pixel_count, percentage))

# Text display
text_str = "Detected Facial Parts:\n\n"
for label_id, part_name, pixel_count, percentage in sorted(detected_parts, key=lambda x: x[3], reverse=True):
    if percentage > 0.1:  # Only show parts with >0.1% coverage
        text_str += f"{part_name:.<30} {percentage:>6.2f}%\n"

axes[1].text(0.05, 0.95, text_str, transform=axes[1].transAxes, fontsize=11,
             verticalalignment="top", fontfamily="monospace",
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
axes[1].axis("off")
axes[1].set_title("Detected Parts & Coverage", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

## 15. Summary

In [ ]:
print("\n" + "#"*70)
print("# HAIR EXTRACTION - SUMMARY")
print("#"*70)
print(f"\n✓ Successfully extracted hair region from image")
print(f"\n  Model: Face Parsing (SegFormer)")
print(f"  Hair pixels detected: {hair_pixel_count:,}")
print(f"  Image coverage: {hair_coverage:.2f}%")
print(f"\n  Available outputs:")
print(f"    • hair_white_bg: Full-size hair with white background")
print(f"    • hair_rgba: Full-size hair with transparency")
print(f"    • hair_cropped_white_bg: Cropped hair (if detected)")
print(f"    • hair_mask: Binary segmentation mask")
print(f"\n" + "#"*70)